# Hosted Tycho Pair Coverage

This notebook visualizes the benchmark pairs selected for the hosted Tycho MVP.

Goal: find token pairs where a significant share of observed volume comes from protocols we can simulate accurately with Tycho, then add our custom protocol as another liquidity source.

In [1]:
from pathlib import Path

import polars as pl
import plotly.graph_objects as go

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)

DATA_CANDIDATES = [Path("data/processed"), Path("../data/processed")]
DATA = next(path for path in DATA_CANDIDATES if (path / "token_pair_coverage.parquet").exists())
SELECTED_PAIRS = [
    "USDC-WETH",
    "USDT-WETH",
    "WBTC-WETH",
    "USDT-WBTC",
    "USDC-WBTC",
    "LINK-WETH",
    "DAI-WETH",
    "AAVE-USDC",
    "AAVE-WETH",
    "UNI-WETH",
]

HOSTED_COVERED = {
    "uniswap_v2",
    "sushiswap_v2",
    "pancakeswap_v2",
    "uniswap_v3",
    "pancakeswap_v3",
    "uniswap_v4",
    "ekubo_v2",
    "ekubo_v3",
    "fluid_v1",
}

MISSING_MAJOR = {"curve", "balancer_v2", "balancer_v3_uncovered"}

pairs = (
    pl.read_parquet(DATA / "token_pair_coverage.parquet")
    .with_columns((pl.col("token0_symbol") + "-" + pl.col("token1_symbol")).alias("pair"))
    .filter(pl.col("pair").is_in(SELECTED_PAIRS))
)

In [2]:
summary_rows = []
for pair in SELECTED_PAIRS:
    df = pairs.filter(pl.col("pair") == pair)
    total = df["volume_30d_usd"].sum()
    hosted = df.filter(pl.col("protocol").is_in(HOSTED_COVERED))["volume_30d_usd"].sum()
    missing_major = df.filter(pl.col("protocol").is_in(MISSING_MAJOR))["volume_30d_usd"].sum()
    summary_rows.append({
        "pair": pair,
        "volume_30d_usd": total,
        "volume_30d_millions": total / 1_000_000,
        "hosted_covered_volume_30d_usd": hosted,
        "hosted_coverage_pct": hosted / total * 100 if total else 0,
        "curve_balancer_pct": missing_major / total * 100 if total else 0,
    })

summary = pl.DataFrame(summary_rows)
summary

pair,volume_30d_usd,volume_30d_millions,hosted_covered_volume_30d_usd,hosted_coverage_pct,curve_balancer_pct
str,f64,f64,f64,f64,f64
"""USDC-WETH""",5.1842e9,5184.158791,5.0906e9,98.194986,1.805014
"""USDT-WETH""",2.2737e9,2273.74685,2.2170e9,97.506043,2.493957
"""WBTC-WETH""",8.7766e8,877.656625,8.6318e8,98.350465,1.649535
"""USDT-WBTC""",7.4314e8,743.136412,7.0879e8,95.377862,4.622138
"""USDC-WBTC""",2.6101e8,261.007014,2.4333e8,93.228218,6.771782
"""LINK-WETH""",1.0547e8,105.468214,1.0546e8,99.996381,0.003619
"""DAI-WETH""",8.0354e7,80.354325,7.9710e7,99.197789,0.802211
"""AAVE-USDC""",3.6709e7,36.708713,3.6709e7,99.999762,0.000238
"""AAVE-WETH""",2.5359e7,25.35889,2.5359e7,99.99998,0.00002


In [3]:
chart = summary.sort("volume_30d_usd")
fig = go.Figure()
fig.add_bar(
    x=chart["volume_30d_usd"].to_list(),
    y=chart["pair"].to_list(),
    orientation="h",
    marker_color="#2563eb",
    text=[f"{v:.1f}% covered" for v in chart["hosted_coverage_pct"].to_list()],
    textposition="outside",
    hovertemplate="%{y}<br>30d volume: $%{x:,.0f}<extra></extra>",
)
fig.update_layout(
    title="Recommended MVP Benchmark Pairs: 30d Volume and Hosted Tycho Coverage",
    height=520,
    xaxis_title="30d volume USD",
    yaxis_title="Pair",
    xaxis_tickprefix="$",
    xaxis_tickformat="~s",
    margin=dict(l=90, r=120, t=70, b=40),
)
fig.show()

In [4]:
composition = (
    pairs
    .with_columns([
        pl.when(pl.col("protocol").is_in(HOSTED_COVERED))
        .then(pl.col("protocol"))
        .otherwise(pl.lit("not_hosted_or_not_enabled"))
        .alias("coverage_bucket")
    ])
    .group_by(["pair", "coverage_bucket"])
    .agg(pl.col("volume_30d_usd").sum())
    .join(summary.select(["pair", "volume_30d_usd"]).rename({"volume_30d_usd": "pair_total_30d_usd"}), on="pair")
    .with_columns((pl.col("volume_30d_usd") / pl.col("pair_total_30d_usd") * 100).alias("pair_share_pct"))
)

protocols = composition.sort("coverage_bucket")["coverage_bucket"].unique().to_list()
fig = go.Figure()
for protocol in protocols:
    values = []
    for pair in SELECTED_PAIRS:
        cell = composition.filter((pl.col("pair") == pair) & (pl.col("coverage_bucket") == protocol))
        values.append(cell["pair_share_pct"].sum() if cell.height else 0)
    fig.add_bar(name=protocol, x=SELECTED_PAIRS, y=values, hovertemplate="%{x}<br>%{fullData.name}: %{y:.2f}%<extra></extra>")

fig.update_layout(
    title="Protocol Composition by Pair",
    barmode="stack",
    height=560,
    yaxis_title="Share of pair volume (%)",
    xaxis_title="Pair",
    yaxis_ticksuffix="%",
    legend_title_text="Protocol",
    margin=dict(l=70, r=40, t=70, b=110),
)
fig.update_xaxes(tickangle=35)
fig.show()

In [5]:
top3 = (
    pairs
    .filter(pl.col("protocol").is_in(HOSTED_COVERED))
    .join(summary.select(["pair", "volume_30d_usd"]).rename({"volume_30d_usd": "pair_total_30d_usd"}), on="pair")
    .with_columns((pl.col("volume_30d_usd") / pl.col("pair_total_30d_usd") * 100).alias("pair_share_pct"))
    .sort(["pair", "volume_30d_usd"], descending=[False, True])
    .with_columns(pl.col("volume_30d_usd").rank("ordinal", descending=True).over("pair").alias("rank"))
    .filter(pl.col("rank") <= 3)
    .select(["pair", "rank", "protocol", "volume_30d_usd", "pair_share_pct"])
)

top3

pair,rank,protocol,volume_30d_usd,pair_share_pct
str,u32,str,f64,f64
"""AAVE-USDC""",1,"""uniswap_v4""",3.5931e7,97.881057
"""AAVE-USDC""",2,"""uniswap_v3""",777748.564563,2.118703
"""AAVE-USDC""",3,"""pancakeswap_v2""",0.492351,0.000001
"""AAVE-WETH""",1,"""uniswap_v3""",2.5332e7,99.895001
"""AAVE-WETH""",2,"""uniswap_v2""",26617.283556,0.104962
"""AAVE-WETH""",3,"""sushiswap_v2""",4.352099,0.000017
"""DAI-WETH""",1,"""sushiswap_v2""",4.0075e7,49.873292
"""DAI-WETH""",2,"""uniswap_v3""",3.0529e7,37.993292
"""DAI-WETH""",3,"""uniswap_v2""",9.1051e6,11.331205


In [6]:
protocols = top3.sort("protocol")["protocol"].unique().to_list()
fig = go.Figure()
for protocol in protocols:
    values = []
    labels = []
    for pair in SELECTED_PAIRS:
        cell = top3.filter((pl.col("pair") == pair) & (pl.col("protocol") == protocol))
        if cell.height:
            values.append(cell["pair_share_pct"].sum())
            labels.append(protocol)
        else:
            values.append(0)
            labels.append("")
    fig.add_bar(
        name=protocol,
        x=SELECTED_PAIRS,
        y=values,
        text=labels,
        textposition="inside",
        hovertemplate="%{x}<br>%{fullData.name}: %{y:.2f}%<extra></extra>",
    )

fig.update_layout(
    title="Top 3 Hosted-Tycho-Covered Protocols per Pair",
    barmode="stack",
    height=560,
    yaxis_title="Share of pair volume (%)",
    xaxis_title="Pair",
    yaxis_ticksuffix="%",
    legend_title_text="Protocol",
    margin=dict(l=70, r=40, t=70, b=110),
)
fig.update_xaxes(tickangle=35)
fig.show()

## Hosted Tycho Validation

The selected pairs were also validated against `tycho-fynd-ethereum.propellerheads.xyz` using the current hosted API plan and plan-required filters: `TOKEN_MIN_QUALITY=100`, `MAX_DAYS_SINCE_LAST_TRADE=3`, and `TVL_GT=10`.

All ten recommended benchmark pairs returned at least one live quote from hosted Tycho. The validation details and best observed quotes are recorded in `../reports/hosted_tycho_validation.md`.

## Interpretation

- `USDC-WETH`, `USDT-WETH`, `WBTC-WETH`, and `USDT-WBTC` are the strongest large-volume benchmark pairs.
- `USDC-WBTC` is still useful, but missing Curve/Balancer share is higher than the ETH pairs.
- `DAI-WETH` is useful because it is not dominated by Curve in this dataset and has meaningful Sushi/Uniswap split.
- `LINK`, `AAVE`, and `UNI` pairs are smaller but clean protocol-token benchmarks with almost all volume in hosted-Tycho-coverable protocols.